# NB-Step4 · U-Net Retraining on Aligned Ground Truth
**Pipeline position:** Step 4 of 9 — runs after NB-Step3, produces a retrained U-Net model.

### What changed from the original training
| | Original | This notebook |
|---|---|---|
| Patch source | Raw 4K frames | Preprocessed PNGs (NB-Step2) |
| Annotation coordinates | Original 2160×3840 space | Remapped to 1080×1475 space (NB-Step3) |
| Mask generation | Synthesised ellipses from bbox | Actual CVAT segmentation polygons |
| pos_weight | 14 (heuristic) | 16 (computed from bbox area statistics) |
| Training images | 490 ROI-valid patches | 859 fully aligned patches |

### Key numbers
- Images: 147 → train 125 / val 22 (image-level split, no leakage)
- Annotations: 859 → ~730 positive patches + ~1095 negative patches
- Patch size: 32×32 px (bubble fills 6–20% of patch area)
- Output frame size: 1080×1475 px

### Outputs
| File | Description |
|------|-------------|
| `unet_aligned.pth` | Best-validation-loss model weights |
| `training_report_<ts>.json` | Per-epoch loss and metric history |
| `training_curves_<ts>.png` | Loss + IoU curves |


In [ ]:
# ── Cell 1 · Imports ─────────────────────────────────────────────────────────
import cv2, json, os, random, time, copy
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Imports complete")
print(f"  PyTorch {torch.__version__}  |  Device: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("  ⚠  No GPU detected — training will be slow. "
          "Runtime → Change runtime type → T4 GPU.")


In [ ]:
# ── Cell 2 · Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✓ Drive mounted")


In [ ]:
# ── Cell 3 · Static Configuration ───────────────────────────────────────────
# HUMAN-EDITED section.  Hyperparameters are computed from data — only paths
# should need changing.

ANNOT_BASE    = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/anot_pool"
COCO_PREPROC  = f"{ANNOT_BASE}/in_df_147_01_preproc.json"
PREPROC_FRAMES = f"{ANNOT_BASE}/preproc_frames_147"
MODEL_DIR     = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/models"

os.makedirs(MODEL_DIR, exist_ok=True)
STEP4_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── Patch parameters ──────────────────────────────────────────────────────────
PATCH_SIZE  = 32       # px — bubble fills 6-20% of patch (median bbox 7.2×10.7 px)
NEG_RATIO   = 1.5      # negative patches per positive
NEG_MIN_DIST = 20      # min distance (px) from any annotation centre for neg patches
RANDOM_SEED = 42

# ── Training hyperparameters ──────────────────────────────────────────────────
TRAIN_SPLIT  = 0.85    # image-level — 125 train / 22 val
BATCH_SIZE   = 16
MAX_EPOCHS   = 200
PATIENCE     = 30      # early stopping on val loss
LR           = 1e-3
LR_PATIENCE  = 10      # ReduceLROnPlateau
LR_FACTOR    = 0.5

# Loss weights (computed from data in NB-Step3 output)
# pos_weight = (1 - fg_fraction) / fg_fraction
# fg_fraction = median_bubble_area / patch_area = 60 / 1024 ≈ 0.059
POS_WEIGHT   = 16.0
DICE_W       = 0.5
BCE_W        = 0.5

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("✓ Cell 3 — configuration loaded")
print(f"  COCO input    : {COCO_PREPROC}")
print(f"  Model output  : {MODEL_DIR}")
print(f"  Patch size    : {PATCH_SIZE}×{PATCH_SIZE} px")
print(f"  pos_weight    : {POS_WEIGHT}")
print(f"  Max epochs    : {MAX_EPOCHS}  |  Patience: {PATIENCE}")


In [ ]:
# ── Cell 4 · Load Data & Train/Val Split ─────────────────────────────────────

with open(COCO_PREPROC) as f:
    coco = json.load(f)

# Build image_id → {png_path, annotations, width, height}
annots_by_img = defaultdict(list)
for ann in coco["annotations"]:
    annots_by_img[ann["image_id"]].append(ann)

image_records = []
missing = []
for img in coco["images"]:
    png_path = os.path.join(PREPROC_FRAMES, img["file_name"])
    if not os.path.exists(png_path):
        missing.append(img["file_name"])
        continue
    image_records.append({
        "image_id":   img["id"],
        "file_name":  img["file_name"],
        "png_path":   png_path,
        "width":      img["width"],
        "height":     img["height"],
        "annotations": annots_by_img[img["id"]],
    })

if missing:
    print(f"⚠  {len(missing)} PNG(s) not found — check PREPROC_FRAMES path.")
    for m in missing[:5]: print(f"   {m}")

# Image-level train/val split (stratified by run_label via filename)
random.shuffle(image_records)
n_train = int(len(image_records) * TRAIN_SPLIT)
train_images = image_records[:n_train]
val_images   = image_records[n_train:]

n_train_ann = sum(len(r["annotations"]) for r in train_images)
n_val_ann   = sum(len(r["annotations"]) for r in val_images)

print(f"✓ Data loaded")
print(f"  Total images      : {len(image_records)}  ({len(missing)} missing)")
print(f"  Train images      : {len(train_images)}  ({n_train_ann} annotations)")
print(f"  Val   images      : {len(val_images)}  ({n_val_ann} annotations)")
print(f"  Ann/image (train) : {n_train_ann/len(train_images):.1f}")


In [ ]:
# ── Cell 5 · Patch Generation ────────────────────────────────────────────────
# Extracts 32×32 patches + binary masks from preprocessed PNGs.
# Positive patches: centred on each annotation bbox centre.
# Negative patches: random positions with no annotation within NEG_MIN_DIST px.

HALF = PATCH_SIZE // 2

def poly_to_mask(segmentation, patch_x0, patch_y0, patch_size):
    """Render a COCO segmentation polygon into a patch-local binary mask."""
    mask = np.zeros((patch_size, patch_size), dtype=np.uint8)
    for poly in segmentation:
        pts = np.array(poly).reshape(-1, 2)
        pts_local = pts - np.array([patch_x0, patch_y0])
        cv2.fillPoly(mask, [pts_local.astype(np.int32)], 1)
    return mask


def extract_patch(frame, cx, cy, patch_size):
    """Extract a patch centred on (cx, cy), zero-padding at borders."""
    half  = patch_size // 2
    H, W  = frame.shape[:2]
    x0, y0 = int(cx) - half, int(cy) - half
    x1, y1 = x0 + patch_size, y0 + patch_size
    # Compute valid region
    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(W, x1), min(H, y1)
    patch = np.zeros((patch_size, patch_size), dtype=np.float32)
    dx0, dy0 = sx0 - x0, sy0 - y0
    dx1, dy1 = dx0 + (sx1 - sx0), dy0 + (sy1 - sy0)
    patch[dy0:dy1, dx0:dx1] = frame[sy0:sy1, sx0:sx1].astype(np.float32) / 255.0
    return patch, x0, y0


def build_patches(records, neg_ratio, neg_min_dist, patch_size, desc=""):
    patches, masks, labels = [], [], []
    half = patch_size // 2
    n_load_fail = 0
    for rec in records:
        try:
          with open(rec["png_path"], 'rb') as fh:
              raw = np.frombuffer(fh.read(), dtype=np.uint8)
          frame = cv2.imdecode(raw, cv2.IMREAD_GRAYSCALE)
        except Exception:
          frame = None
        if frame is None:
            n_load_fail += 1
            continue
        H, W = frame.shape
        anns  = rec["annotations"]
        if not anns:
            continue

        # Annotation centres
        centres = [(a["bbox"][0] + a["bbox"][2]/2,
                    a["bbox"][1] + a["bbox"][3]/2) for a in anns]

        # ── Positive patches ────────────────────────────────────────────────
        for ann, (cx, cy) in zip(anns, centres):
            if (cx < half or cy < half or
                    cx > W - half or cy > H - half):
                continue   # too close to border for a full patch
            patch, px0, py0 = extract_patch(frame, cx, cy, patch_size)
            mask             = poly_to_mask(ann["segmentation"],
                                            px0, py0, patch_size)
            patches.append(patch)
            masks.append(mask.astype(np.float32))
            labels.append(1)

        # ── Negative patches ────────────────────────────────────────────────
        n_neg     = int(len(anns) * neg_ratio)
        generated = 0
        attempts  = 0
        max_att   = n_neg * 20

        while generated < n_neg and attempts < max_att:
            attempts += 1
            nx = random.randint(half, W - half)
            ny = random.randint(half, H - half)
            # Reject if too close to any annotation centre
            if any(abs(nx - cx) < neg_min_dist and
                   abs(ny - cy) < neg_min_dist
                   for cx, cy in centres):
                continue
            patch, _, _ = extract_patch(frame, nx, ny, patch_size)
            mask         = np.zeros((patch_size, patch_size), dtype=np.float32)
            patches.append(patch)
            masks.append(mask)
            labels.append(0)
            generated += 1

    pos = sum(labels)
    neg = len(labels) - pos
    print(f"  {desc:<8}  {len(patches):>5} patches  "
          f"(pos={pos}  neg={neg}  ratio={neg/max(pos,1):.1f}x)")
    if n_load_fail:
        print(f"  ⚠  {n_load_fail} frame(s) failed to load")
    return patches, masks, labels


print("Generating patches …")
t0 = time.time()
tr_patches, tr_masks, tr_labels = build_patches(
    train_images, NEG_RATIO, NEG_MIN_DIST, PATCH_SIZE, "TRAIN")
va_patches, va_masks, va_labels = build_patches(
    val_images, NEG_RATIO, NEG_MIN_DIST, PATCH_SIZE, "VAL")
print(f"Done ({time.time()-t0:.1f}s)")


In [ ]:
print(f"\n  Val images with zero patches: "
      f"{sum(1 for r in val_images if not any(True for a in r['annotations']))}")

In [ ]:
# ── Cell 6 · PyTorch Dataset & DataLoaders ───────────────────────────────────

class PatchDataset(Dataset):
    """Grayscale 32×32 patch dataset with optional augmentation."""
    def __init__(self, patches, masks, augment=False):
        self.patches = patches
        self.masks   = masks
        self.augment = augment

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        p = self.patches[idx].copy()   # (H, W) float32 0-1
        m = self.masks[idx].copy()     # (H, W) float32 0/1

        if self.augment:
            # Horizontal flip
            if random.random() > 0.5:
                p, m = np.fliplr(p).copy(), np.fliplr(m).copy()
            # Vertical flip
            if random.random() > 0.5:
                p, m = np.flipud(p).copy(), np.flipud(m).copy()
            # 90° rotation (0/1/2/3 × 90°)
            k = random.randint(0, 3)
            p, m = np.rot90(p, k).copy(), np.rot90(m, k).copy()

        p = torch.from_numpy(p).unsqueeze(0)   # (1, H, W)
        m = torch.from_numpy(m).unsqueeze(0)   # (1, H, W)
        return p, m


train_ds = PatchDataset(tr_patches, tr_masks, augment=True)
val_ds   = PatchDataset(va_patches, va_masks, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"✓ Datasets ready")
print(f"  Train : {len(train_ds)} patches  →  {len(train_loader)} batches")
print(f"  Val   : {len(val_ds)} patches  →  {len(val_loader)} batches")


In [ ]:
# ── Cell 7 · U-Net Model ─────────────────────────────────────────────────────
# Exact architecture from the paper: encoder features [16, 32], two MaxPool
# levels.  Input 1×32×32 → output 1×32×32 sigmoid probability map.

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c,  out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, features=(16, 32)):
        super().__init__()
        f1, f2 = features
        self.enc1       = ConvBlock(1,      f1)
        self.pool1      = nn.MaxPool2d(2)
        self.enc2       = ConvBlock(f1,     f2)
        self.pool2      = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(f2,     f2 * 2)
        self.up2        = nn.ConvTranspose2d(f2 * 2, f2, 2, stride=2)
        self.dec2       = ConvBlock(f2 * 2, f2)
        self.up1        = nn.ConvTranspose2d(f2,     f1, 2, stride=2)
        self.dec1       = ConvBlock(f1 * 2, f1)
        self.out_conv   = nn.Conv2d(f1, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b  = self.bottleneck(self.pool2(e2))
        d2 = self.dec2(torch.cat([self.up2(b),  e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.out_conv(d1))


def dice_loss(pred, target, smooth=1e-6):
    p = pred.view(-1)
    t = target.view(-1)
    return 1 - (2 * (p * t).sum() + smooth) / (p.sum() + t.sum() + smooth)


def combined_loss(pred, target, pos_weight, dice_w=0.5, bce_w=0.5):
    bce  = F.binary_cross_entropy(pred, target, reduction='none')
    wmap = target * (pos_weight - 1) + 1
    bce  = (bce * wmap).mean()
    dice = dice_loss(pred, target)
    return bce_w * bce + dice_w * dice


def patch_iou(pred, target, threshold=0.5):
    p = (pred > threshold).float().view(-1)
    t = target.view(-1)
    inter = (p * t).sum()
    union = p.sum() + t.sum() - inter
    return (inter / (union + 1e-6)).item()


model     = UNet(features=(16, 32)).to(DEVICE)
optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode='min',
                               patience=LR_PATIENCE, factor=LR_FACTOR,)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ U-Net instantiated")
print(f"  Trainable parameters : {n_params:,}")
print(f"  Loss : {BCE_W}×BCE(pos_weight={POS_WEIGHT}) + {DICE_W}×Dice")


In [ ]:
# ── Cell 8 · Training Loop ───────────────────────────────────────────────────

history = {"train_loss": [], "val_loss": [], "val_iou": [], "lr": []}
best_val_loss = float("inf")
best_weights  = None
patience_ctr  = 0

print(f"Training for up to {MAX_EPOCHS} epochs  "
      f"(early stopping patience={PATIENCE})\n")
print(f"  {'Ep':>4}  {'TrainLoss':>10}  {'ValLoss':>10}  "
      f"{'ValIoU':>8}  {'LR':>10}  Note")
print(f"  {'─'*4}  {'─'*10}  {'─'*10}  {'─'*8}  {'─'*10}  {'─'*10}")

t_start = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    if len(train_ds) == 0:
          raise RuntimeError(
              "train_ds is empty — no patches were generated.\n"
              "Re-run Cell 5 after applying the imread fix."
          )

    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    tr_loss = 0.0
    for patches, masks in train_loader:
        patches = patches.to(DEVICE)
        masks   = masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(patches)
        loss  = combined_loss(preds, masks, POS_WEIGHT, DICE_W, BCE_W)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * len(patches)
    tr_loss /= len(train_ds)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    va_loss = 0.0
    va_iou  = 0.0
    with torch.no_grad():
        for patches, masks in val_loader:
            patches = patches.to(DEVICE)
            masks   = masks.to(DEVICE)
            preds   = model(patches)
            loss    = combined_loss(preds, masks, POS_WEIGHT, DICE_W, BCE_W)
            va_loss += loss.item() * len(patches)
            va_iou  += patch_iou(preds, masks) * len(patches)
    va_loss /= max(len(val_ds), 1)
    va_iou  /= max(len(val_ds), 1)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(va_loss)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["val_iou"].append(va_iou)
    history["lr"].append(current_lr)

    # ── Early stopping ────────────────────────────────────────────────────────
    note = ""
    if va_loss < best_val_loss:
        best_val_loss = va_loss
        best_weights  = copy.deepcopy(model.state_dict())
        patience_ctr  = 0
        note = "★ best"
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

    # Print every 10 epochs or on improvement
    if epoch % 10 == 0 or note:
        elapsed = time.time() - t_start
        print(f"  {epoch:>4}  {tr_loss:>10.4f}  {va_loss:>10.4f}  "
              f"{va_iou:>8.4f}  {current_lr:>10.2e}  {note}")

total_epochs = len(history["train_loss"])
print(f"\nTraining complete — {total_epochs} epochs  "
      f"({time.time()-t_start:.0f}s)")
print(f"  Best val loss : {best_val_loss:.4f}")
print(f"  Best val IoU  : {max(history['val_iou']):.4f}")


In [ ]:
# ── Cell 9 · Training Curves ─────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs, history["train_loss"], label="Train loss", color="steelblue")
axes[0].plot(epochs, history["val_loss"],   label="Val loss",   color="tomato")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Combined Loss (BCE + Dice)")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["val_iou"], label="Val IoU", color="seagreen")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("IoU")
axes[1].set_title("Validation IoU @ threshold 0.5")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
curves_path = os.path.join(MODEL_DIR, f"training_curves_{STEP4_TS}.png")
plt.savefig(curves_path, dpi=120, bbox_inches="tight")
plt.show(); plt.close()
print(f"✓ Training curves saved: {curves_path}")


In [ ]:
# ── Cell 10 · Save Model & Training Report ───────────────────────────────────

# Restore best weights
model.load_state_dict(best_weights)

model_path = os.path.join(MODEL_DIR, "unet_aligned.pth")
torch.save({
    "model_state_dict": best_weights,
    "model_config":     {"features": (16, 32), "patch_size": PATCH_SIZE},
    "best_val_loss":    best_val_loss,
    "best_val_iou":     max(history["val_iou"]) if history["val_iou"] else 0.0,
    "total_epochs":     len(history["train_loss"]),
    "step4_ts":         STEP4_TS,
    "hyperparams": {
        "patch_size":  PATCH_SIZE,
        "pos_weight":  POS_WEIGHT,
        "dice_w":      DICE_W,
        "bce_w":       BCE_W,
        "lr":          LR,
        "batch_size":  BATCH_SIZE,
        "neg_ratio":   NEG_RATIO,
    },
}, model_path)
print(f"✓ Model saved: {model_path}")

report = {
    "generated_at":    STEP4_TS,
    "model_path":      model_path,
    "coco_input":      COCO_PREPROC,
    "n_train_images":  len(train_images),
    "n_val_images":    len(val_images),
    "n_train_patches": len(train_ds),
    "n_val_patches":   len(val_ds),
    "total_epochs":    len(history["train_loss"]),
    "best_val_loss":   best_val_loss,
    "best_val_iou":    max(history["val_iou"]),
    "history":         history,
}
report_path = os.path.join(MODEL_DIR, f"training_report_{STEP4_TS}.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"✓ Training report saved: {report_path}")

W = 66
print()
print("=" * W)
print("  NB-Step4 · SUMMARY".center(W))
print("=" * W)
print(f"  Train patches  : {len(train_ds)}  "
      f"(pos={sum(tr_labels)}  neg={len(tr_labels)-sum(tr_labels)})")
print(f"  Val patches    : {len(val_ds)}")
print(f"  Epochs trained : {len(history['train_loss'])}")
print(f"  Best val loss  : {best_val_loss:.4f}")
print(f"  Best val IoU   : {max(history['val_iou']):.4f}" if history['val_iou'] else "  Best val IoU   : n/a (empty val set)")
print()
print(f"  Model          : {model_path}")
print()
print("  ✓ Step 4 complete.")
print("  Next → NB-Step5: re-evaluate segmentation coverage on full inference frames.")
print("  Key input: unet_aligned.pth")
print("=" * W)


In [ ]:
import json, os

MODEL_DIR = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/models"

# Load most recent training report
reports = sorted([f for f in os.listdir(MODEL_DIR)
                  if f.startswith("training_report_") and f.endswith(".json")])
with open(os.path.join(MODEL_DIR, reports[-1])) as f:
    rep = json.load(f)

print(f"Total epochs      : {rep['total_epochs']}")
print(f"Best val loss     : {rep['best_val_loss']:.4f}")
print(f"Best val IoU      : {rep['best_val_iou']:.4f}")
print(f"Train patches     : {rep['n_train_patches']}")
print(f"Val patches       : {rep['n_val_patches']}")
print(f"Train images      : {rep['n_train_images']}")
print(f"Val images        : {rep['n_val_images']}")

# Final epoch loss
hist = rep["history"]
print(f"\nFinal train loss  : {hist['train_loss'][-1]:.4f}")
print(f"Final val loss    : {hist['val_loss'][-1]:.4f}")
print(f"Final val IoU     : {hist['val_iou'][-1]:.4f}")
print(f"Min val loss      : {min(hist['val_loss']):.4f}  "
      f"at epoch {hist['val_loss'].index(min(hist['val_loss']))+1}")